# Customer Support Agent — Interactive Demo

This notebook walks through the LangGraph-powered customer support agent. The agent can handle billing queries, technical questions, complaints (with automatic ticket creation), and escalation to a human agent via a **human-in-the-loop** interrupt.

Each example uses a separate `thread_id` so conversation state is isolated per session.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from src.graph import graph
from langgraph.types import Command

load_dotenv()

## Example 1: Billing Query

In [ ]:
config = {"configurable": {"thread_id": "demo-billing"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="Why was I charged twice this month?")]},
    config=config,
)

print(result["messages"][-1].content)

## Example 2: Technical Question

In [ ]:
config = {"configurable": {"thread_id": "demo-tech"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="How do I reset my password?")]},
    config=config,
)

print(result["messages"][-1].content)

## Example 3: Complaint → Ticket Creation

In [ ]:
config = {"configurable": {"thread_id": "demo-complaint"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="Your service has been down for 3 days, this is unacceptable")]},
    config=config,
)

print(result["messages"][-1].content)
print("Ticket ID:", result.get("ticket_id"))

## Example 4: Human-in-the-Loop Escalation

When a customer requests a human agent, the graph pauses at an **interrupt** node and waits for a human operator to provide a response. We then resume execution by passing a `Command(resume=...)` with the agent's reply.

In [ ]:
config = {"configurable": {"thread_id": "demo-escalate"}}

# First invocation — the graph will pause waiting for a human agent response
result = graph.invoke(
    {"messages": [HumanMessage(content="I need to speak to a human agent")]},
    config=config,
)

# The graph has paused at the human-in-the-loop interrupt node.
# In a production system a human operator would now be notified.
print("Graph paused. Last message:", result["messages"][-1].content)

In [ ]:
# Resume execution with the human agent's reply
result = graph.invoke(
    Command(resume="Hi, I'm Sarah from the support team. How can I help you today?"),
    config=config,
)

print(result["messages"][-1].content)

## Visualizing the Graph

LangGraph can render the compiled graph as a Mermaid diagram, giving a clear view of all nodes and edges.

In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_mermaid_png())